In [1]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")


In [2]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

In [3]:
train_df = train_df.sort_values(by="timestamp_num").reset_index(drop=True)
valid_df = valid_df.sort_values(by="timestamp_num").reset_index(drop=True)
test_df = test_df.sort_values(by="timestamp_num").reset_index(drop=True)

In [4]:
train_df = train_df.sort_values(by=['timestamp_num'])
valid_df = valid_df.sort_values(by=['timestamp_num'])
test_df = test_df.sort_values(by=['timestamp_num'])

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Kiểm tra GPU có sẵn cho TensorFlow không
print(f"TensorFlow phiên bản: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Số lượng GPU khả dụng cho TensorFlow: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print("Trạng thái GPU:", gpu)

# ---------------------------------------------------------
# 1. TÁCH DỮ LIỆU & ĐỊNH DẠNG ĐẦU VÀO
# ---------------------------------------------------------
print("Chuẩn bị dữ liệu...")
# Lấy X, y từ các tập dữ liệu gốc (không chia nhỏ train thêm nữa)
X_train = train_df.drop(columns=['label']).values
y_train = train_df['label'].values

X_valid = valid_df.drop(columns=['label']).values
y_valid = valid_df['label'].values

X_test = test_df.drop(columns=['label']).values
y_test = test_df['label'].values

num_classes = len(np.unique(y_train))
print(f"Số lượng classes: {num_classes}")

# Reshape X sang dạng 3D (samples, time_steps=1, features) cho LSTM/GRU
X_train_3d = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_valid_3d = X_valid.reshape(X_valid.shape[0], 1, X_valid.shape[1])
X_test_3d = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])


# ---------------------------------------------------------
# 2. XÂY DỰNG VÀ HUẤN LUYỆN CÁC MÔ HÌNH CƠ SỞ (TẦNG 1)
# ---------------------------------------------------------
print("\n--- Bắt đầu huấn luyện Tầng 1 (Level-1 Base Models) trên TOÀN BỘ TẬP TRAIN ---")

# 1. Gradient Boosting
print("- Đang huấn luyện GBM (HistGradientBoosting trên CPU)...")
gbm = HistGradientBoostingClassifier()
gbm.fit(X_train, y_train)

# 2. LightGBM
print("- Đang huấn luyện LightGBM (trên CPU đa luồng)...")
lgbm = LGBMClassifier(verbose=-1, n_jobs=-1)
lgbm.fit(X_train, y_train)

# 3. Bagging (Giữ nguyên tối ưu RAM: max_samples=0.01, max_depth=15, n_jobs=2)
print("- Đang huấn luyện Bagging (1000 cây trên CPU)... (Có thể mất vài phút)")
dt_base = DecisionTreeClassifier(max_depth=15)
bagging = BaggingClassifier(estimator=dt_base, n_estimators=1000, max_samples=0.01, n_jobs=2)
bagging.fit(X_train, y_train)

# 4. XGBoost (Chạy GPU)
print("- Đang huấn luyện XGBoost (trên GPU)...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', tree_method='hist', device='cuda')
xgb.fit(X_train, y_train)

# 5. CatBoost (Chạy GPU)
print("- Đang huấn luyện CatBoost (trên GPU)...")
cat = CatBoostClassifier(verbose=0, task_type='GPU')
cat.fit(X_train, y_train)

# Thêm EarlyStopping sử dụng tự tách 10% data bên trong nội bộ tập Train
nn_es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# 6. LSTM (Chạy GPU tự động)
print("- Đang huấn luyện LSTM (trên GPU)...")
lstm_model = Sequential([
    LSTM(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
lstm_model.fit(X_train_3d, y_train, validation_split=0.1, epochs=10, batch_size=1024, callbacks=[nn_es], verbose=1)

# 7. GRU (Chạy GPU tự động)
print("- Đang huấn luyện GRU (trên GPU)...")
gru_model = Sequential([
    GRU(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])
gru_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
gru_model.fit(X_train_3d, y_train, validation_split=0.1, epochs=10, batch_size=1024, callbacks=[nn_es], verbose=1)


# ---------------------------------------------------------
# 3. TẠO CÁC ĐẶC TRƯNG META TỪ TẬP VALID (HOLD-OUT) VÀ TEST
# ---------------------------------------------------------
print("\n--- Trích xuất Meta-Features từ Tầng 1 ---")
def get_meta_features(X_2d, X_3d):
    p_gbm = gbm.predict_proba(X_2d)
    p_lgbm = lgbm.predict_proba(X_2d)
    p_bag = bagging.predict_proba(X_2d)
    p_xgb = xgb.predict_proba(X_2d)
    p_cat = cat.predict_proba(X_2d)
    p_lstm = lstm_model.predict(X_3d, verbose=0)
    p_gru = gru_model.predict(X_3d, verbose=0)
    
    return np.hstack([p_gbm, p_lgbm, p_bag, p_xgb, p_cat, p_lstm, p_gru])

print("- Tạo dữ liệu huấn luyện Meta-Model trên tập VALID (Hold-out)...")
X_meta_train = get_meta_features(X_valid, X_valid_3d)

print("- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...")
X_meta_test = get_meta_features(X_test, X_test_3d)


# ---------------------------------------------------------
# 4. HUẤN LUYỆN META-MODEL (TẦNG 2) VÀ ĐÁNH GIÁ TRÊN TEST
# ---------------------------------------------------------
print("\n--- Huấn luyện Meta-Model (EnsembleGuard Neural Network) Tầng 2 ---")
meta_model_nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_meta_train.shape[1],)),
    Dense(num_classes, activation='softmax')
])

meta_model_nn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Huấn luyện trên tập meta validation (X_meta_train, y_valid)
meta_model_nn.fit(X_meta_train, y_valid, epochs=20, batch_size=1024, verbose=1)


print("\n==============================================")
print("ĐÁNH GIÁ ENSEMBLE TRÊN TẬP TEST ĐỘC LẬP TỪ META-MODEL")
print("==============================================")

meta_preds_proba = meta_model_nn.predict(X_meta_test)
meta_preds_classes = np.argmax(meta_preds_proba, axis=1)

print("\n--- Báo cáo Phân loại (Classification Report) ---")
print(classification_report(y_test, meta_preds_classes, digits=4))

acc = accuracy_score(y_test, meta_preds_classes)
prec = precision_score(y_test, meta_preds_classes, average='macro', zero_division=0)
rec = recall_score(y_test, meta_preds_classes, average='macro', zero_division=0)
f1 = f1_score(y_test, meta_preds_classes, average='macro', zero_division=0)

print(f"\n=> Tỷ lệ chính xác (Accuracy): {acc:.4f}")
print(f"=> Macro Precision:          {prec:.4f}")
print(f"=> Macro Recall:             {rec:.4f}")
print(f"=> Macro F1-Score:           {f1:.4f}")

TensorFlow phiên bản: 2.19.0
Số lượng GPU khả dụng cho TensorFlow: 0
Chuẩn bị dữ liệu...
Số lượng classes: 11

--- Bắt đầu huấn luyện Tầng 1 (Level-1 Base Models) trên TOÀN BỘ TẬP TRAIN ---
- Đang huấn luyện GBM (HistGradientBoosting trên CPU)...
- Đang huấn luyện LightGBM (trên CPU đa luồng)...
- Đang huấn luyện Bagging (1000 cây trên CPU)... (Có thể mất vài phút)
- Đang huấn luyện XGBoost (trên GPU)...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


- Đang huấn luyện CatBoost (trên GPU)...
- Đang huấn luyện LSTM (trên GPU)...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 40s 18ms/step - loss: 0.2679 - val_loss: 0.0622
Epoch 2/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 39s 17ms/step - loss: 0.1248 - val_loss: 0.0772
Epoch 3/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - loss: 0.1202 - val_loss: 0.1216
Epoch 4/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 40s 17ms/step - loss: 0.1214 - val_loss: 0.0961
- Đang huấn luyện GRU (trên GPU)...
Epoch 1/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 31s 14ms/step - loss: 0.2543 - val_loss: 0.1420
Epoch 2/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.1263 - val_loss: 0.0247
Epoch 3/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.1184 - val_loss: 0.0345
Epoch 4/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.1165 - val_loss: 0.0918
Epoch 5/10
2172/2172 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.1148 - val_loss: 0.0622

--- Trích xuất Meta-Features từ Tầng 1 ---
- Tạo dữ liệu huấn luyện Meta-Model trên tập VALID (Hold-out)...


...

- Tạo dữ liệu đánh giá Meta-Model trên tập TEST...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Huấn luyện Meta-Model (EnsembleGuard Neural Network) Tầng 2 ---
Epoch 1/20


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


557/557 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8849 - loss: 0.6796
Epoch 2/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9796 - loss: 0.0557
Epoch 3/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9811 - loss: 0.0514
Epoch 4/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9819 - loss: 0.0493
Epoch 5/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9822 - loss: 0.0481
Epoch 6/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9823 - loss: 0.0473
Epoch 7/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9828 - loss: 0.0460
Epoch 8/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9829 - loss: 0.0452
Epoch 9/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9832 - loss: 0.0444
Epoch 10/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9829 - loss: 0.0445
Epoch 11/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9833 - loss: 0.0437
Epoch 12/20
557/557 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy